# FLAG 2027 - Notebook 16: LoRA fine-tuning of WavLM - PILOT (the big swing)

All frozen-feature tricks are exhausted (Notebooks 11, 15). The FAME 2024 winner gained ~6 EER points from fine-tuning its encoders cross-modally. Here we let **WavLM-base-plus adapt its attention layers (LoRA, ~0.6M trainable weights)** while training the face-voice head end to end.

- Voice input = [LoRA-WavLM embedding (trained) + cached ECAPA + ResNet293 + WavLM-sv]; face input = cached CLIP. Same FOP head, CE + OPL.
- Data: challenge + MAV-Celeb v1/v3 (184 in-domain people; raw audio needed, so no VoxCeleb here).
- WavLM-base-plus was pretrained on English/European data only (LibriLight, GigaSpeech, VoxPopuli) and we fine-tune only on challenge + MAV-Celeb audio (English/Urdu/German) -> no Bengali (rule-safe).
- **PILOT = 2 folds x 1 seed**, compared with the SAME folds of N4 (identical data, frozen features, Notebook 12).

**Decision rule:** continue to the full version only if LoRA beats N4 by >= 1 point on BOTH folds.

About 1-1.5 h. Paste the **SUMMARY** to Claude.

In [ ]:
# Cell 1 - Drive, GPU, packages
# Colab ships an old 'torchao' (0.10) that the new 'peft' refuses; we do not need it -> remove it
from google.colab import drive
drive.mount('/content/drive')
!pip -q uninstall -y torchao
!pip -q install peft
import torch, shutil
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', dev, '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))

In [ ]:
# Cell 2 - unpack the wavs (challenge train + MAV v1/v3, same row order as our feature files) and cache 6-s waveforms
import os, re, json, time, zipfile, copy
import numpy as np, pandas as pd
import librosa
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027')
FEATDIR = os.path.join(PROJECT, 'features')
RESDIR = os.path.join(PROJECT, 'results')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
MAV_ZIPS, MAV_LISTS = os.path.join(BASE, 'dataset'), os.path.join(BASE, 'face_voice_association_splits')
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)

FULL = '/content/flag_data'; os.makedirs(FULL, exist_ok=True)
if not os.path.exists(os.path.join(FULL, 'train.done')):
    with zipfile.ZipFile(os.path.join(ZIP_DIR, 'train_set.zip')) as zf:
        zf.extractall(FULL, members=[m for m in zf.namelist() if m.endswith(('.wav', '.txt', '.csv'))])
    open(os.path.join(FULL, 'train.done'), 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(FULL) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def resolve(rel, list_path):
    q = os.path.dirname(list_path)
    while q.startswith(FULL):
        c = os.path.join(q, rel)
        if os.path.exists(c):
            return c
        q = os.path.dirname(q)
tl = find_one('train_English.txt')
train_lines = [l.split() for l in open(tl) if l.strip()]
WAV = [resolve(t[2], tl) for t in train_lines]
assert None not in WAV

def zip_index(zf):
    # same rule as Notebook 06: map 'voices/...' and 'faces/...' to the zip member name
    m = {}
    for n in zf.namelist():
        for key in ('voices/', 'faces/'):
            i = n.find(key)
            if i >= 0 and not n.endswith('/'):
                m[n[i:]] = n; break
    return m
MAVD = '/content/mav'; mspk = []
for ver, zname, lists in [('v1', 'mavceleb_v1_train.zip', ['English_train.txt', 'Urdu_train.txt']),
                          ('v3', 'mavceleb_v3_tran.zip', ['English_train.txt', 'German_train.txt'])]:
    flag = os.path.join(MAVD, ver + '.voices.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); zi = zip_index(zf)
    for lf in lists:
        for l in open(os.path.join(MAV_LISTS, ver, lf)):
            t = l.split()
            if len(t) < 2:
                continue
            vm, fm = zi.get('voices/' + t[0]), zi.get('faces/' + t[1])
            if vm is None or fm is None:          # Notebook 06 kept a row only if BOTH files exist
                continue
            p = os.path.join(MAVD, ver, vm)
            if need:
                zf.extract(vm, os.path.join(MAVD, ver))
            WAV.append(p); mspk.append(ver + '_' + t[0].split('/')[0])
    zf.close()
    if need:
        open(flag, 'w').close()
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz')))
if list(MAV['spk']) != mspk:
    raise SystemExit('MAV row order differs (%d vs %d) - tell Claude' % (len(mspk), len(MAV['spk'])))
N_CH, N_MAV = len(train_lines), len(MAV['spk'])
note('wav files: challenge %d + MAV %d' % (N_CH, N_MAV))

SEC = 6; L = 16000 * SEC
def load6(p):
    w = librosa.load(p, sr=16000, mono=True)[0]
    if len(w) >= L:
        s0 = (len(w) - L) // 2; w = w[s0:s0 + L]
    else:
        w = np.tile(w, int(np.ceil(L / max(len(w), 1))))[:L]
    return (np.clip(w, -1, 1) * 32767).astype(np.int16)
CACHE6 = '/content/wav6.npy'
if not os.path.exists(CACHE6):
    t0 = time.time()
    arr = np.lib.format.open_memmap(CACHE6, mode='w+', dtype=np.int16, shape=(len(WAV), L))
    with ThreadPoolExecutor(8) as ex:
        for i, w in enumerate(tqdm(ex.map(load6, WAV), total=len(WAV), desc='decode 6-s waveforms')):
            arr[i] = w
    arr.flush(); del arr
    note('waveform cache built in %.0f min' % ((time.time() - t0) / 60))
WAVE = np.load(CACHE6, mmap_mode='r')

In [ ]:
# Cell 3 - cached features (face CLIP; voice ECAPA + WavLM-sv + ResNet293), labels, folds (identical to Notebook 12)
import torch.nn as nn
import torch.nn.functional as Fn
from sklearn.metrics import roc_curve
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
Xv, y = read_feat(find_one('train_English_voices.csv'), 192); y = y.astype(int)
CL = np.load(os.path.join(FEATDIR, 'clip.npz')); WL = np.load(os.path.join(FEATDIR, 'wavlm_sv.npz')); R2 = np.load(os.path.join(FEATDIR, 'r293.npz'))
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
F_RAW = np.concatenate([l2(CL['train']), l2(MAV['clip'])]).astype(np.float32)
V_RAW = np.concatenate([np.concatenate([l2(Xv), l2(WL['train']), l2(R2['train'])], 1),
                        np.concatenate([l2(MAV['ecapa']), l2(MAV['wavlm_sv']), l2(R2['mav'])], 1)]).astype(np.float32)
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
mav_lab = {s: 1000 + i for i, s in enumerate(sorted(set(MAV['spk'])))}
Y_ALL = np.r_[y, np.array([mav_lab[s] for s in MAV['spk']])]
D_ALL = np.r_[np.zeros(N_CH, int), np.ones(N_MAV, int)]
EXT = N_CH + np.arange(N_MAV)
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed)
    spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, Lb = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); Lb.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); Lb.append(0)
    return np.array(V), np.array(F), np.array(Lb)
K = 5
rng = np.random.default_rng(42)
folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k])
    va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, va_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
print('ready: rows', len(Y_ALL), '| voice cached dim', V_RAW.shape[1])

In [ ]:
# Cell 4 - model: LoRA-WavLM + cached features -> FOP head
from transformers import WavLMModel
from peft import LoraConfig, get_peft_model

class LoraVoice(nn.Module):
    def __init__(self, r=8):
        super().__init__()
        base = WavLMModel.from_pretrained('microsoft/wavlm-base-plus')
        base.feature_extractor._freeze_parameters()
        cfg = LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.1, target_modules=['q_proj', 'v_proj'])
        self.wavlm = get_peft_model(base, cfg)
        self.proj = nn.Sequential(nn.Linear(768, 256), nn.LayerNorm(256))
    def forward(self, wav):                      # wav: (B, T) float in [-1, 1]
        wav = (wav - wav.mean(1, keepdim=True)) / (wav.std(1, keepdim=True) + 1e-6)
        h = self.wavlm(wav).last_hidden_state   # (B, frames, 768)
        return self.proj(h.mean(1))

class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class Model(nn.Module):
    def __init__(self, n_class, d_face, d_voice_cached, d=128):
        super().__init__()
        self.lv = LoraVoice()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice_cached + 256, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class)
    def embed(self, xf, xv, wav):
        f = self.face(xf); v = self.voice(torch.cat([xv, self.lv(wav).float()], 1))
        return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv, wav):
        ft, vt, f, v = self.embed(xf, xv, wav)
        a = torch.sigmoid(self.att(torch.cat([f, v], 1)))
        fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused)
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t()
    same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
m_ = LoraVoice(); n_tr = sum(p.numel() for p in m_.parameters() if p.requires_grad); del m_
print('trainable weights in LoRA voice tower: %.2f M' % (n_tr / 1e6))

In [ ]:
# Cell 5 - PILOT: folds 0 and 1, seed 1; compare with N4 (frozen, same data) on the same folds
P_ID, K_CLIP, ALPHA, CROP = 16, 4, 2.0, 16000 * 3      # 64 clips per step, random 3-s crops for training
MAX_EP, STEPS, EVAL_EVERY, PATIENCE = 30, 60, 2, 4

def norm_stats(rows):
    F, V = F_RAW.copy(), V_RAW.copy(); ch = None
    for d in np.unique(D_ALL[rows]):
        trd = rows[D_ALL[rows] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_RAW[trd].mean(0), F_RAW[trd].std(0) + 1e-6, V_RAW[trd].mean(0), V_RAW[trd].std(0) + 1e-6)
        F[alld] = (F_RAW[alld] - st[0]) / st[1]; V[alld] = (V_RAW[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch

@torch.no_grad()
def voice_embed_rows(model, rows, XV, bs=32):
    model.eval(); out = {}
    for i in range(0, len(rows), bs):
        r = rows[i:i + bs]
        wav = torch.tensor(WAVE[r].astype(np.float32) / 32767, device=dev)
        with torch.autocast('cuda', dtype=torch.float16):
            v = model.voice(torch.cat([XV[torch.tensor(r, device=dev)], model.lv(wav).float()], 1))
        for j, rr in enumerate(r):
            out[int(rr)] = torch.tanh(v[j].float())
    return out

@torch.no_grad()
def evaluate(model, XF, XV, va_idx, vsets):
    vemb = voice_embed_rows(model, va_idx, XV)
    model.eval()
    res = []
    for Vi, Fi, Lb in vsets:
        ft = torch.tanh(model.face(XF[torch.tensor(Fi, device=dev)]))
        vt = torch.stack([vemb[int(r)] for r in Vi])
        res.append(eer(Lb, -(ft - vt).norm(dim=1).cpu().numpy()))
    return res

def train_fold(k, seed=1):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    tr_idx, va_idx, vsets = FOLD_DATA[k]
    all_tr = np.concatenate([tr_idx, EXT])
    F, V, ch = norm_stats(all_tr)
    XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev)
    pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    model = Model(len(cl), F.shape[1], V.shape[1]).to(dev)
    lora_p = [p for n, p in model.lv.named_parameters() if p.requires_grad]
    head_p = [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith('lv.')]
    opt = torch.optim.AdamW([{'params': lora_p, 'lr': 5e-5}, {'params': head_p, 'lr': 1e-4}], weight_decay=0.01)
    scaler = torch.amp.GradScaler('cuda')
    best, best_ep, best_state, wait, t0 = 1e9, 0, None, 0, time.time()
    for ep in range(1, MAX_EP + 1):
        model.train()
        for step in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j = np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen])
            rows = all_tr[j]
            s0 = rng.integers(0, L - CROP + 1, len(rows))
            wav = np.stack([WAVE[r, s:s + CROP] for r, s in zip(rows, s0)]).astype(np.float32) / 32767
            rt = torch.tensor(rows, device=dev)
            with torch.autocast('cuda', dtype=torch.float16):
                fused, logits = model(XF[rt], XV[rt], torch.tensor(wav, device=dev))
                loss = Fn.cross_entropy(logits.float(), ytr[torch.tensor(j, device=dev)]) + ALPHA * opl_loss(fused.float(), ytr[torch.tensor(j, device=dev)])
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            if ep == 1 and step == 19:
                print('  speed: %.2f s/step' % ((time.time() - t0) / 20))
        if ep % EVAL_EVERY:
            continue
        e = evaluate(model, XF, XV, va_idx, vsets); sc = np.mean(e)
        print('  fold %d epoch %2d | std %.2f gender %.2f | %.0f min' % (k, ep, e[0], e[1], (time.time() - t0) / 60))
        if sc < best - 0.05:
            best, best_ep, best_state, wait = sc, ep, {n: p.detach().clone() for n, p in model.state_dict().items()}, 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    model.load_state_dict(best_state)
    e = evaluate(model, XF, XV, va_idx, vsets)
    return e, best_ep

CSV = os.path.join(RESDIR, 'nb16_pilot.csv')
done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['fold'])
for k in [0, 1]:
    if (done.fold == k).any():
        continue
    e, be = train_fold(k)
    pd.DataFrame([dict(fold=k, best_ep=be, std=e[0], gender=e[1])]).to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False)
    done = pd.read_csv(CSV)

N4 = pd.read_csv(os.path.join(RESDIR, 'nb12_cv.csv'))
N4 = N4[(N4.combo == 'N4_clip|ecapa+wavlm+r293') & (N4.seed == 1)].set_index('fold')
rows = []
for _, r in pd.read_csv(CSV).iterrows():
    k = int(r.fold); b = N4.loc[k]
    rows.append(dict(fold=k, lora_std=r['std'], lora_gender=r['gender'], lora_overall=(r['std'] + r['gender']) / 2,
                     N4_std=b['std'], N4_gender=b['gender'], N4_overall=(b['std'] + b['gender']) / 2))
T = pd.DataFrame(rows).round(2)
T['gain'] = (T.N4_overall - T.lora_overall).round(2)
note('LoRA-WavLM PILOT vs N4 (same folds, same data, seed 1); gain > 0 means LoRA is better')
note(T.to_string(index=False))
note('DECISION: ' + ('GO - LoRA beats N4 by >= 1 on both folds' if (T.gain >= 1).all() else 'NO-GO (or unclear) - keep frozen features'))
print('=' * 70)
print('SUMMARY  (copy everything between the lines and paste to Claude)')
print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)